# Predicting Isotopic Ratios from Major and Trace Element Chemistry

## What We're Doing

**The problem:** Isotope measurements (Sr, Nd, Pb) are expensive and time-consuming. PetDB has ~500,000 samples with major element chemistry, but only ~50,000 have isotope data.

**The idea:** Can we predict isotopes from the chemistry that's already measured? If so, we could "fill in" isotope values for hundreds of thousands of samples.

**Why it might work:**
- Isotope ratios reflect long-term element fractionation (Rb/Sr → ⁸⁷Sr/⁸⁶Sr)
- Incompatible element enrichment often correlates with radiogenic enrichment
- Different tectonic settings have characteristic chemistry AND isotope signatures

**What we'll do:**
1. Load PetDB data (isotopes + major/trace elements)
2. Explore correlations between chemistry and isotopes
3. Train ML models (XGBoost, neural networks) to predict isotopes
4. Use SHAP to understand which elements matter most
5. Honestly assess where the model works and where it fails

**Success criteria:** R² > 0.7 for at least one isotope system, with interpretable feature importance.


---

## Section 1: Introduction & Motivation

### Why Isotopes Matter

Radiogenic isotope systems—particularly **Sr**, **Nd**, and **Pb**—are among the most powerful tools in igneous petrology for tracing magma sources and evolution:

- **⁸⁷Sr/⁸⁶Sr** reflects time-integrated Rb/Sr ratios, distinguishing depleted (low) from enriched (high) sources
- **¹⁴³Nd/¹⁴⁴Nd** (often expressed as εNd) tracks Sm/Nd evolution, typically anticorrelated with Sr isotopes
- **²⁰⁶Pb/²⁰⁴Pb**, **²⁰⁷Pb/²⁰⁴Pb**, **²⁰⁸Pb/²⁰⁴Pb** record U-Th-Pb fractionation history

These systems allow geochemists to:
1. **Identify mantle reservoirs**: MORB-source depleted mantle (DMM) vs. enriched mantle (EM1, EM2) vs. HIMU
2. **Detect crustal contamination**: Continental crust has distinct isotopic signatures
3. **Constrain mixing processes**: Isotopes don't fractionate during melting or crystallization

### The Data Gap Problem

Despite their utility, isotope ratios are measured far less frequently than major and trace elements:

| Data Type | Typical PetDB Coverage | Relative Cost |
|-----------|----------------------|---------------|
| Major elements | ~500,000 samples | $ |
| Trace elements | ~200,000 samples | $$ |
| Isotope ratios | ~20,000 samples | $$$$ |

**Why the gap?**
- TIMS/MC-ICP-MS instrumentation is expensive and requires specialized labs
- Sample preparation (chemical separation) is labor-intensive
- Analysis time per sample is much longer than XRF/ICP-MS
- Many labs lack isotope capabilities entirely

### Why Machine Learning Might Work

The physical basis for predictability:

1. **Parent-daughter relationships**: Present-day isotope ratios depend on parent/daughter element ratios. Rb/Sr directly affects ⁸⁷Sr/⁸⁶Sr evolution.

2. **Correlated fractionation**: Elements that behave similarly during magmatic processes tend to correlate. Incompatible element enrichment (high K₂O, LREE) often accompanies radiogenic enrichment.

3. **Tectonic associations**: MORB samples cluster in isotope space differently from OIB or arc samples—and these settings have characteristic trace element patterns.

4. **Non-linear relationships**: ML models can capture complex, multivariate relationships that simple regressions miss.

### What We're Attempting

**Goal**: Train ML models on the ~20,000 samples with both chemistry AND isotopes, then predict isotopes for the ~480,000 samples with only chemistry.

**Key questions**:
1. How accurately can isotopes be predicted from major/trace elements?
2. Which features are most predictive (and do they make geochemical sense)?
3. Where does the model fail, and can we quantify prediction uncertainty?
4. Can this tool meaningfully "fill in" the global isotope database?

---

## Section 2: Data Acquisition

We use pre-downloaded and pre-joined data from PetDB containing:
1. Isotope data (Sr, Nd, Pb ratios)
2. Major element data (SiO₂, MgO, etc.)
3. Trace element data (Rb, Sr, REE, etc.)

The raw data was queried from PetDB's WFS API and joined on `SamplingFeatureURI`.

In [26]:
# Setup and imports
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import time
from pathlib import Path

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# PetDB GCS Mirror (primary data source)
PETDB_GCS_MIRROR = "gs://wustl-eeps-edc/petdb-mirror"

# Local paths for outputs
FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

print("Imports complete.")
print(f"Data source: {PETDB_GCS_MIRROR}")


Imports complete.
Data source: gs://wustl-eeps-edc/petdb-mirror


### 2.1 Load Pre-Joined Dataset

The dataset was pre-processed to:
- Join isotope samples with matching major/trace element data via `SamplingFeatureURI`
- Aggregate duplicate analyses per sample (mean values)
- Extract lat/lon from geometry
- Keep only relevant columns (~55 MB in memory vs ~400+ MB for raw GeoDataFrames)

In [27]:
# Load data from GCS mirror
print("Loading PetDB data from GCS mirror...")
print("="*50)

start = time.time()

# Load isotopes
print("Loading isotopes...")
isotopes_df = pd.read_parquet(f"{PETDB_GCS_MIRROR}/isotopes.parquet")
print(f"  {len(isotopes_df):,} rows")

# Load major elements  
print("Loading major elements...")
majors_df = pd.read_parquet(f"{PETDB_GCS_MIRROR}/major_elements.parquet")
print(f"  {len(majors_df):,} rows")

# Load trace elements
print("Loading trace elements...")
traces_df = pd.read_parquet(f"{PETDB_GCS_MIRROR}/trace_elements.parquet")
print(f"  {len(traces_df):,} rows")

load_time = time.time() - start
print(f"\nTotal load time: {load_time:.1f} seconds")


Loading PetDB data from GCS mirror...
Loading isotopes...
  52,417 rows
Loading major elements...
  333,883 rows
Loading trace elements...
  463,440 rows

Total load time: 5.0 seconds


In [28]:
# Join datasets on samplingfeatureuri
print("Joining datasets...")
print("="*50)

# Get isotope sample URIs
iso_uris = set(isotopes_df['samplingfeatureuri'].dropna())
print(f"Unique isotope sample URIs: {len(iso_uris):,}")

# Identify column types
# Isotope columns (lowercase)
isotope_cols = ['sr87_sr86', 'nd143_nd144', 'epsilon_nd', 'pb206_pb204', 'pb207_pb204', 'pb208_pb204']
isotope_cols = [c for c in isotope_cols if c in isotopes_df.columns]
print(f"Isotope columns: {isotope_cols}")

# Major element columns (no _wtpct suffix in GCS data)
major_oxide_names = ['sio2', 'tio2', 'al2o3', 'fe2o3', 'feo', 'mno', 'mgo', 'cao', 'na2o', 'k2o', 'p2o5', 'fe2o3_total', 'feo_total']
major_cols = [c for c in major_oxide_names if c in majors_df.columns]
print(f"Major element columns: {major_cols}")

# Trace element columns
trace_names = ['rb', 'sr', 'ba', 'nb', 'zr', 'y', 'la', 'ce', 'nd', 'sm', 'eu', 'gd', 'dy', 'er', 'yb', 'lu', 'th', 'u', 'pb', 'hf', 'ta']
trace_cols = [c for c in traces_df.columns if any(c.lower() == t or c.lower().endswith(f'_{t}') or c.lower() == f'{t}_ppm' for t in trace_names)]
trace_cols = trace_cols[:20]  # Limit for memory
print(f"Trace element columns: {len(trace_cols)}")

# Filter majors/traces to isotope samples only
majors_filtered = majors_df[majors_df['samplingfeatureuri'].isin(iso_uris)]
traces_filtered = traces_df[traces_df['samplingfeatureuri'].isin(iso_uris)]
print(f"\nMajors matching isotope samples: {len(majors_filtered):,}")
print(f"Traces matching isotope samples: {len(traces_filtered):,}")

# Aggregate to one row per sample (mean of duplicates)
majors_agg = majors_filtered.groupby('samplingfeatureuri')[major_cols].mean().reset_index()
print(f"Unique samples with majors: {len(majors_agg):,}")

if trace_cols:
    traces_agg = traces_filtered.groupby('samplingfeatureuri')[trace_cols].mean().reset_index()
    print(f"Unique samples with traces: {len(traces_agg):,}")

# Build combined dataset starting from isotopes
meta_cols = ['samplingfeatureuri', 'sample_id', 'specimentype', 'materialclass', 
             'rockname', 'reference', 'latitude', 'longitude']
meta_cols = [c for c in meta_cols if c in isotopes_df.columns]

df = isotopes_df[meta_cols + isotope_cols].copy()
df = df.merge(majors_agg, on='samplingfeatureuri', how='left')
if trace_cols:
    df = df.merge(traces_agg, on='samplingfeatureuri', how='left')

# Summary
n_with_majors = df[major_cols].notna().any(axis=1).sum()
print(f"\n{'='*50}")
print(f"Combined dataset: {len(df):,} samples")
print(f"  With isotopes: {len(df):,}")
print(f"  With major elements: {n_with_majors:,}")
print(f"{'='*50}")

# Free memory
del isotopes_df, majors_df, traces_df, majors_filtered, traces_filtered
import gc; gc.collect()


Joining datasets...


KeyError: 'SamplingFeatureURI'

In [ ]:
# Examine dataset structure
print("Columns in dataset:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Identify column types
isotope_cols = ['Sr87_Sr86', 'Nd143_Nd144', 'Epsilon_Nd', 'Pb206_Pb204', 'Pb207_Pb204', 'Pb208_Pb204']
isotope_cols = [c for c in isotope_cols if c in df.columns]

major_cols = [c for c in df.columns if 'WtPct' in c]
trace_cols = [c for c in df.columns if '_ppm' in c.lower()]
meta_cols = ['SamplingFeatureURI', 'SpecimenID', 'SpecimenType', 'MaterialClass', 'RockName', 'Citation', 'latitude', 'longitude']
meta_cols = [c for c in meta_cols if c in df.columns]

print(f"Isotope columns ({len(isotope_cols)}): {isotope_cols}")
print(f"\nMajor element columns ({len(major_cols)}): {major_cols}")
print(f"\nTrace element columns ({len(trace_cols)}): {trace_cols[:10]}...")
print(f"\nMetadata columns ({len(meta_cols)}): {meta_cols}")

### 2.2 Data Quality Summary

In [ ]:
# Summarize data coverage
print("="*60)
print("DATA COVERAGE SUMMARY")
print("="*60)

print(f"\nTotal samples: {len(df):,}")

print("\nIsotope coverage:")
for col in isotope_cols:
    if col in df.columns:
        n = df[col].notna().sum()
        pct = 100 * n / len(df)
        print(f"  {col}: {n:,} ({pct:.1f}%)")

print("\nMajor element coverage:")
n_any_major = df[major_cols].notna().any(axis=1).sum() if major_cols else 0
print(f"  Any major element: {n_any_major:,} ({100*n_any_major/len(df):.1f}%)")

if trace_cols:
    print("\nTrace element coverage:")
    available_traces = [c for c in trace_cols if c in df.columns]
    if available_traces:
        n_any_trace = df[available_traces].notna().any(axis=1).sum()
        print(f"  Any trace element: {n_any_trace:,} ({100*n_any_trace/len(df):.1f}%)")


In [ ]:
# Calculate oxide totals for quality filtering
oxide_sum_cols = ['SiO2_WtPct', 'TiO2_WtPct', 'Al2O3_WtPct', 'FeOt_WtPct', 'Fe2O3t_WtPct',
                  'MnO_WtPct', 'MgO_WtPct', 'CaO_WtPct', 'Na2O_WtPct', 'K2O_WtPct', 'P2O5_WtPct']
oxide_sum_cols = [c for c in oxide_sum_cols if c in df.columns]

if oxide_sum_cols:
    df['oxide_total'] = df[oxide_sum_cols].sum(axis=1, skipna=True)
    
    print("Oxide total distribution:")
    print(df['oxide_total'].describe())
    
    # Filter to reasonable totals (98-102%)
    mask_good_total = (df['oxide_total'] >= 98) & (df['oxide_total'] <= 102)
    print(f"\nSamples with oxide totals 98-102%: {mask_good_total.sum():,} ({100*mask_good_total.mean():.1f}%)")

### 2.3 Create Analysis Dataset

Filter to samples that have:
1. At least one isotope measurement
2. Major element data (for prediction features)

In [ ]:
# Create analysis dataset
has_isotope = df[isotope_cols].notna().any(axis=1)
has_majors = df[major_cols].notna().any(axis=1) if major_cols else pd.Series(True, index=df.index)

# For modeling, we need both
analysis_mask = has_isotope & has_majors
df_analysis = df[analysis_mask].copy()

print(f"Analysis dataset: {len(df_analysis):,} samples")
print(f"  (Samples with both isotopes AND major elements)")

# Store as our working dataset
df = df_analysis

---

## Section 3: Exploratory Data Analysis

Before building models, we need to understand our data:
1. How do elements correlate with isotopes?
2. Is our training data geographically representative?
3. What does the data look like in isotope space?

### 3.1 Correlation Matrix

Which major/trace elements correlate most strongly with isotope ratios?

In [ ]:
def plot_correlation_heatmap(df: pd.DataFrame, figsize: tuple = (14, 12)) -> go.Figure:
    """
    Create a correlation heatmap between elements and isotopes.
    """
    # Select columns for correlation
    corr_cols = []
    
    # Major oxides (no _wtpct suffix)
    for oxide in ['sio2', 'tio2', 'al2o3', 'mgo', 'cao', 'na2o', 'k2o', 'feo_total']:
        if oxide in df.columns:
            corr_cols.append(oxide)
    
    # Isotopes
    for iso in ['sr87_sr86', 'nd143_nd144', 'epsilon_nd', 'pb206_pb204']:
        if iso in df.columns:
            corr_cols.append(iso)
    
    if len(corr_cols) < 3:
        print(f"Not enough columns for correlation. Found: {corr_cols}")
        print(f"Available columns: {df.columns.tolist()[:20]}...")
        return None
    
    print(f"Computing correlations for: {corr_cols}")
    
    # Compute correlation
    subset = df[corr_cols].apply(pd.to_numeric, errors='coerce')
    corr = subset.corr()
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.columns,
        colorscale='RdBu_r',
        zmid=0,
        text=np.round(corr.values, 2),
        texttemplate='%{text}',
        textfont={'size': 10},
        hovertemplate='%{x} vs %{y}<br>r = %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Correlation Matrix: Major Elements vs Isotopes',
        width=figsize[0]*72,
        height=figsize[1]*72,
        yaxis_autorange='reversed'
    )
    
    return fig


fig = plot_correlation_heatmap(df)
if fig:
    fig.show()
    fig.write_html(FIGURES / 'correlation_heatmap.html')
    print(f"Saved to {FIGURES / 'correlation_heatmap.html'}")


### 3.2 Geographic Distribution

Where are our training samples located? Are certain regions over- or under-represented?

In [ ]:
def plot_sample_map(df: pd.DataFrame, color_col: str = None) -> go.Figure:
    """
    Create an interactive map of sample locations.
    """
    if 'latitude' not in df.columns or 'longitude' not in df.columns:
        print("No lat/lon columns found.")
        return None
    
    # Prepare data
    plot_df = df[['latitude', 'longitude']].dropna().copy()
    
    if color_col and color_col in df.columns:
        plot_df['color'] = pd.to_numeric(df.loc[plot_df.index, color_col], errors='coerce')
        plot_df = plot_df.dropna(subset=['color'])
    
    print(f"Plotting {len(plot_df):,} samples with valid coordinates")
    
    # Create map
    if 'color' in plot_df.columns:
        fig = px.scatter_geo(
            plot_df,
            lat='latitude',
            lon='longitude',
            color='color',
            color_continuous_scale='Viridis',
            title=f'Training Data Distribution (colored by {color_col})',
            projection='natural earth'
        )
    else:
        fig = px.scatter_geo(
            plot_df,
            lat='latitude',
            lon='longitude',
            title='Training Data Distribution',
            projection='natural earth'
        )
    
    fig.update_traces(marker=dict(size=4, opacity=0.5))
    fig.update_layout(
        width=1000,
        height=600,
        geo=dict(
            showland=True,
            landcolor='lightgray',
            showocean=True,
            oceancolor='lightblue',
            coastlinecolor='darkgray'
        )
    )
    
    return fig


fig = plot_sample_map(df)
if fig:
    fig.show()
    fig.write_html(FIGURES / 'sample_map.html')
    print(f"Saved to {FIGURES / 'sample_map.html'}")


### 3.3 Isotope Space Visualization

Traditional isotope diagrams (Sr-Nd, Pb-Pb) help visualize mantle end-members and mixing trends.

In [ ]:
# Sr-Nd diagram
if 'sr87_sr86' in df.columns and 'nd143_nd144' in df.columns:
    color_col = 'mgo' if 'mgo' in df.columns else None
    fig = plot_isotope_diagram(
        df,
        x_col='sr87_sr86',
        y_col='nd143_nd144',
        color_col=color_col,
        title='Sr-Nd Isotope Diagram' + (f' (colored by MgO)' if color_col else '')
    )
    if fig:
        fig.show()
        fig.write_html(FIGURES / 'sr_nd_diagram.html')
        print(f"Saved to {FIGURES / 'sr_nd_diagram.html'}")
else:
    print("Sr-Nd diagram requires both isotope columns.")
    print(f"Available: {[c for c in df.columns if 'sr' in c.lower() or 'nd' in c.lower()]}")


### 3.4 Summary Statistics by Isotope System

In [ ]:
def isotope_statistics(df: pd.DataFrame, isotope_cols: list) -> pd.DataFrame:
    """
    Calculate summary statistics for isotope columns.
    """
    stats = []
    for col in isotope_cols:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(values) > 0:
                stats.append({
                    'Isotope': col,
                    'N': len(values),
                    'Mean': values.mean(),
                    'Std': values.std(),
                    'Min': values.min(),
                    'Max': values.max(),
                })
    
    return pd.DataFrame(stats)


print("\n" + "="*60)
print("ISOTOPE STATISTICS")
print("="*60)
iso_stats = isotope_statistics(df, isotope_cols)
if len(iso_stats) > 0:
    display(iso_stats)
else:
    print("No isotope data found.")


---

## Section 3 Summary

At this point, we have:

1. **Loaded pre-joined data** from PetDB (isotopes + major elements + trace elements)
2. **Assessed data quality** and coverage statistics
3. **Explored correlations** between elements and isotopes
4. **Mapped sample locations** to understand geographic coverage
5. **Visualized isotope space** with traditional diagrams

### Key Findings:

- Dataset contains ~52,000 samples with isotope data
- ~32,000 samples have both isotopes AND major elements (our training set)
- Sr isotopes have the best coverage, followed by Nd and Pb

### Next Steps (Sections 4-6):

1. **Feature Engineering**: Create meaningful element ratios and handle missing data
2. **Model Development**: Train baseline and neural network models
3. **Results & Evaluation**: Assess prediction accuracy for each isotope system

In [ ]:
# Final status check
print("\n" + "="*60)
print("SECTIONS 1-3 COMPLETE")
print("="*60)

print(f"\n✓ Analysis dataset: {len(df):,} samples")
print(f"✓ Isotope systems: {len(isotope_cols)}")
print(f"✓ Major element features: {len(major_cols)}")
print(f"✓ Trace element features: {len(trace_cols)}")
print("\nReady to proceed with feature engineering and modeling.")
